In [0]:
%run ./_common

In [0]:
import os
import time
import requests
from datetime import datetime, timedelta, timezone

log = get_logger("opspulse.ingest")

# Job parameters (defaults are used when run interactively)
dbutils.widgets.text("max_hours", "3")            # max hourly files per run
dbutils.widgets.text("lag_hours", "3")            # skip the newest hours (not published yet)
dbutils.widgets.text("start_hour", "2026-09-20T12")  # used only if landing is empty (UTC)

MAX_HOURS = int(dbutils.widgets.get("max_hours"))
LAG_HOURS = int(dbutils.widgets.get("lag_hours"))
START_HOUR = dbutils.widgets.get("start_hour")
log.info("params: max_hours=%d lag_hours=%d", MAX_HOURS, LAG_HOURS)

In [0]:
def latest_landed_hour():
    """Newest hour that has a finished .json.gz file in the landing zone (UTC), or None."""
    latest = None
    if not os.path.isdir(LANDING_PATH):
        return None
    for date_dir in os.listdir(LANDING_PATH):
        if not date_dir.startswith("date="):
            continue
        day = date_dir.split("=")[1]
        for hour_dir in os.listdir(f"{LANDING_PATH}/{date_dir}"):
            if not hour_dir.startswith("hour="):
                continue
            folder = f"{LANDING_PATH}/{date_dir}/{hour_dir}"
            if not any(f.endswith(".json.gz") for f in os.listdir(folder)):
                continue
            ts = datetime.strptime(day, "%Y-%m-%d").replace(
                hour=int(hour_dir.split("=")[1]), tzinfo=timezone.utc
            )
            latest = ts if latest is None or ts > latest else latest
    return latest


def hours_to_fetch(max_hours: int, lag_hours: int) -> list:
    """Hours after the latest landed one, up to (now - lag), capped at max_hours."""
    latest = latest_landed_hour()
    if latest is None:
        first = datetime.strptime(START_HOUR, "%Y-%m-%dT%H").replace(tzinfo=timezone.utc)
    else:
        first = latest + timedelta(hours=1)
    cutoff = (datetime.now(timezone.utc) - timedelta(hours=lag_hours)).replace(
        minute=0, second=0, microsecond=0
    )
    hours = []
    current = first
    while current <= cutoff and len(hours) < max_hours:
        hours.append(current)
        current += timedelta(hours=1)
    return hours

In [0]:
def download_hour(ts: datetime, retries: int = 3) -> str:
    """Download one hourly file. Returns 'downloaded', 'skipped' or 'not_published'."""
    date = ts.strftime("%Y-%m-%d")
    hour = ts.hour
    url = f"https://data.gharchive.org/{date}-{hour}.json.gz"  # hour has no leading zero
    folder = f"{LANDING_PATH}/date={date}/hour={hour:02d}"
    final = f"{folder}/{date}-{hour}.json.gz"
    tmp = final + ".part"
    os.makedirs(folder, exist_ok=True)

    if os.path.exists(final) and os.path.getsize(final) > 0:
        return "skipped"

    for attempt in range(1, retries + 1):
        try:
            with requests.get(url, stream=True, timeout=60) as r:
                if r.status_code == 404:
                    log.warning("not published yet: %s", url)
                    return "not_published"
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
            os.replace(tmp, final)  # rename only after the full download completes
            log.info("downloaded %s (%.1f MB)", final, os.path.getsize(final) / 1e6)
            return "downloaded"
        except Exception as e:
            log.warning("attempt %d/%d failed for %s: %s", attempt, retries, url, e)
            time.sleep(2 * attempt)
    raise RuntimeError(f"download failed after {retries} attempts: {url}")


def run_ingest() -> dict:
    hours = hours_to_fetch(MAX_HOURS, LAG_HOURS)
    log.info("hours to fetch: %s", [h.strftime("%Y-%m-%d %H") for h in hours])
    result = {"downloaded": 0, "skipped": 0, "not_published": 0}
    for ts in hours:
        status = download_hour(ts)
        result[status] += 1
        if status == "not_published":
            break  # later hours cannot exist either
    log.info("ingest finished: %s", result)
    return result


run_ingest()

In [0]:
rows = []
for date_dir in sorted(os.listdir(LANDING_PATH)):
    if not date_dir.startswith("date="):
        continue
    for hour_dir in sorted(os.listdir(f"{LANDING_PATH}/{date_dir}")):
        folder = f"{LANDING_PATH}/{date_dir}/{hour_dir}"
        files = os.listdir(folder)
        rows.append((date_dir, hour_dir, len(files), any(f.endswith(".part") for f in files)))

print("hour folders:", len(rows))
print("leftover .part files:", sum(1 for r in rows if r[3]))
for r in rows[-6:]:
    print(r)